In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 01: LOCAL COLAB ENVIRONMENT
# ============================================================

import os
import sys
import json
import shutil
import platform
from pathlib import Path

BASE = Path("/content/SARC_Sentinel1")

DIRS = {
    "raw_zips": BASE / "raw_zips",
    "extracted": BASE / "extracted",
    "processed": BASE / "processed",
    "outputs": BASE / "outputs",
    "metadata": BASE / "metadata",
    "temp": Path("/content/SARC_Sentinel1_temp")
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

disk = shutil.disk_usage("/content")

print("=" * 65)
print("SARC SENTINEL-1 ENVIRONMENT INITIALIZED")
print("=" * 65)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project Root:", BASE)

print("\nCOLAB STORAGE")
print(f"Total: {disk.total / 1024**3:.2f} GB")
print(f"Free:  {disk.free / 1024**3:.2f} GB")

print("\nDIRECTORIES")
for name, path in DIRS.items():
    print(f"{name:12s}: {path}")

print("\nUPLOAD LOCATION:")
print(DIRS["raw_zips"])

print("\nSTATUS: READY FOR LOCAL ZIP UPLOAD")
print("=" * 65)


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 02: GOOGLE DRIVE — EXACT TWO SCENES
# ============================================================

from google.colab import drive
from pathlib import Path
import zipfile
import shutil

# Mount Google Drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

# Exact filenames provided
EXPECTED_FILES = [
    "S1A_IW_GRDH_1SDV_20231014T120602_20231014T120627_050761_061DFD_05BE.SAFE.zip",
    "S1A_IW_GRDH_1SDV_20231002T120602_20231002T120627_050586_0617FC_8D3F.SAFE.zip"
]

# Preserve the existing downstream directory structure
RAW = DIRS["raw_zips"]
RAW.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("SEARCHING GOOGLE DRIVE FOR SELECTED SCENES")
print("=" * 65)

located_files = {}

for filename in EXPECTED_FILES:
    matches = list(MYDRIVE.rglob(filename))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"\nZIP NOT FOUND: {filename}\n"
            "Check that the file is present in Google Drive."
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple copies found for {filename}:\n" +
            "\n".join(str(p) for p in matches)
        )

    located_files[filename] = matches[0]
    print(f"\nFOUND: {filename}")
    print("Location:", matches[0])

print("\n" + "=" * 65)
print("COPYING AND VALIDATING ZIP FILES")
print("=" * 65)

total_compressed = 0
total_uncompressed = 0

for filename, source in located_files.items():

    destination = RAW / filename

    if (
        not destination.exists()
        or destination.stat().st_size != source.stat().st_size
    ):
        shutil.copy2(source, destination)

    with zipfile.ZipFile(destination, "r") as z:

        bad_file = z.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"ZIP integrity failed: {filename} | {bad_file}"
            )

        entries = z.infolist()
        uncompressed = sum(item.file_size for item in entries)

    compressed = destination.stat().st_size

    total_compressed += compressed
    total_uncompressed += uncompressed

    print(f"\nScene: {filename}")
    print(f"Compressed size: {compressed / 1024**3:.3f} GB")
    print(f"Uncompressed size: {uncompressed / 1024**3:.3f} GB")
    print("ZIP integrity: PASSED")

print("\n" + "=" * 65)
print("SENTINEL-1 DATA READY")
print("=" * 65)

print("Total scenes:", len(located_files))
print(f"Total compressed: {total_compressed / 1024**3:.2f} GB")
print(f"Total extraction size: {total_uncompressed / 1024**3:.2f} GB")
print("Pre-event: 02 October 2023")
print("Post-event: 14 October 2023")
print("Destination:", RAW)
print("STATUS: READY FOR CELL 03")


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 03: EXTRACT SENTINEL-1 SAFE ARCHIVES
# ============================================================

import zipfile
import shutil
from pathlib import Path

RAW = DIRS["raw_zips"]
EXTRACTED = DIRS["extracted"]

archives = sorted(RAW.glob("*.zip"))

if not archives:
    raise FileNotFoundError("No ZIP archives found. Complete ZIP upload first.")

required_gb = sum(
    sum(item.file_size for item in zipfile.ZipFile(z).infolist())
    for z in archives
) / 1024**3

free_gb = shutil.disk_usage("/content").free / 1024**3

print(f"ZIP archives found: {len(archives)}")
print(f"Estimated extraction size: {required_gb:.2f} GB")
print(f"Available Colab storage: {free_gb:.2f} GB")

if required_gb + 2 > free_gb:
    raise RuntimeError("Insufficient free storage. Do not start extraction.")

for archive in archives:
    destination = EXTRACTED / archive.stem
    destination.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting: {archive.name}")

    with zipfile.ZipFile(archive, "r") as z:
        bad_file = z.testzip()
        if bad_file is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad_file}")

        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise RuntimeError(f"Unsafe ZIP path: {member.filename}")

        z.extractall(destination)

    print(f"Extracted to: {destination}")

safe_folders = list(EXTRACTED.rglob("*.SAFE"))

print("\n" + "=" * 65)
print("SENTINEL-1 EXTRACTION INVENTORY")
print("=" * 65)
print("SAFE folders:", len(safe_folders))

for folder in safe_folders:
    measurement = list(folder.rglob("measurement/*.tiff"))
    annotations = list(folder.rglob("annotation/*.xml"))
    manifest = folder / "manifest.safe"

    print(f"\n{folder.name}")
    print("Measurement TIFFs:", len(measurement))
    print("Annotation XMLs:", len(annotations))
    print("Manifest:", "FOUND" if manifest.exists() else "MISSING")

print("\nExtraction stage completed.")


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 04: GEOSPATIAL DEPENDENCIES
# ============================================================

%pip -q install rasterio geopandas shapely pyproj lxml scipy scikit-learn matplotlib pandas


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 05: AOI AND STUDY CONFIGURATION
# ============================================================

import geopandas as gpd
from shapely.geometry import box
from pyproj import Transformer

# Same Rangpo–Majitar AOI used in the NISAR branch
AOI = {
    "north": 27.2025,
    "south": 27.1575,
    "west": 88.4897,
    "east": 88.5403
}

AOI_CRS = "EPSG:4326"
PROJECT_CRS = "EPSG:32645"

aoi_geometry = box(
    AOI["west"], AOI["south"],
    AOI["east"], AOI["north"]
)

aoi_gdf = gpd.GeoDataFrame(
    {"name": ["Rangpo_Majitar_AOI"]},
    geometry=[aoi_geometry],
    crs=AOI_CRS
).to_crs(PROJECT_CRS)

aoi_path = DIRS["metadata"] / "SARC_Rangpo_Majitar_AOI.geojson"
aoi_gdf.to_file(aoi_path, driver="GeoJSON")

aoi_projected = aoi_gdf.geometry.iloc[0]
print("AOI: Rangpo–Majitar")
print("Geographic bounds:", AOI)
print("Analysis CRS:", PROJECT_CRS)
print("Projected bounds:", aoi_projected.bounds)
print("AOI area:", round(aoi_gdf.geometry.area.iloc[0] / 1e6, 4), "km²")
print("Saved:", aoi_path)


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 06: SAFE METADATA INVENTORY
# ============================================================

from lxml import etree
import pandas as pd
import re

def find_text(root, names):
    for name in names:
        found = root.xpath(f"//*[local-name()='{name}']")
        if found and found[0].text:
            return found[0].text.strip()
    return None

scene_records = []

for safe in sorted(DIRS["extracted"].rglob("*.SAFE")):
    manifest = safe / "manifest.safe"

    if not manifest.exists():
        print("Skipping missing manifest:", safe.name)
        continue

    root = etree.parse(str(manifest))

    record = {
        "scene": safe.name,
        "safe_path": str(safe),
        "platform": find_text(root, ["familyName"]),
        "mode": find_text(root, ["mode"]),
        "product_type": find_text(root, ["productType"]),
        "start_time": find_text(root, ["startTime"]),
        "stop_time": find_text(root, ["stopTime"]),
        "pass": find_text(root, ["pass"]),
        "relative_orbit": find_text(root, ["relativeOrbitNumber"]),
        "absolute_orbit": find_text(root, ["orbitNumber"]),
        "polarizations": sorted(set(
            x.text.strip()
            for x in root.xpath("//*[local-name()='transmitterReceiverPolarisation']")
            if x.text
        )),
        "measurement_files": len(list(safe.rglob("measurement/*.tiff")))
    }

    scene_records.append(record)

scene_df = pd.DataFrame(scene_records)

if scene_df.empty:
    raise RuntimeError("No valid Sentinel-1 SAFE metadata could be read.")

scene_df["start_time"] = pd.to_datetime(scene_df["start_time"], errors="coerce", utc=True)
scene_df["relative_orbit"] = pd.to_numeric(scene_df["relative_orbit"], errors="coerce")

scene_df = scene_df.sort_values("start_time").reset_index(drop=True)

scene_df.to_csv(DIRS["metadata"] / "sentinel1_scene_inventory.csv", index=False)

display(scene_df[
    ["scene", "platform", "mode", "product_type", "start_time",
     "pass", "relative_orbit", "polarizations", "measurement_files"]
])

print("Scene inventory saved.")


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 07: ORBIT AND TEMPORAL COMPATIBILITY
# ============================================================

required_columns = ["start_time", "pass", "relative_orbit", "mode", "product_type"]

for column in required_columns:
    if column not in scene_df.columns:
        raise RuntimeError(f"Missing metadata field: {column}")

scene_df["event_phase"] = scene_df["start_time"].apply(
    lambda t: (
        "PRE_EVENT" if t < pd.Timestamp("2023-10-04", tz="UTC")
        else "POST_EVENT"
    ) if pd.notna(t) else "UNKNOWN"
)

scene_df["acquisition_date"] = scene_df["start_time"].dt.strftime("%Y-%m-%d")

compatibility = (
    scene_df.groupby(["pass", "relative_orbit", "mode", "product_type"], dropna=False)
    .agg(
        scene_count=("scene", "count"),
        first_acquisition=("start_time", "min"),
        last_acquisition=("start_time", "max"),
        dates=("acquisition_date", lambda x: ", ".join(x.dropna()))
    )
    .reset_index()
)

print("EVENT DATE: 04 OCTOBER 2023")
print("\nSCENE TIMELINE")
display(scene_df[["scene", "acquisition_date", "event_phase", "pass", "relative_orbit"]])

print("\nORBIT COMPATIBILITY GROUPS")
display(compatibility)

scene_df.to_csv(DIRS["metadata"] / "sentinel1_event_timeline.csv", index=False)
compatibility.to_csv(DIRS["metadata"] / "sentinel1_orbit_groups.csv", index=False)


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 08: ANNOTATION-BASED GEOLOCATION AND AOI INTERSECTION
# ============================================================

import rasterio
import pandas as pd
import numpy as np
from lxml import etree
from shapely.geometry import MultiPoint

footprint_records = []

for _, row in scene_df.iterrows():

    safe = Path(row["safe_path"])
    measurement_files = sorted(safe.rglob("measurement/*.tiff"))
    annotation_dir = safe / "annotation"

    for raster_path in measurement_files:

        # Determine polarization from measurement filename
        name_parts = raster_path.stem.lower().split("-")
        polarization = next(
            (p.upper() for p in name_parts if p in ["vv", "vh", "hh", "hv"]),
            None
        )

        if polarization is None:
            print("Polarization not identified:", raster_path.name)
            continue

        try:
            # Find corresponding annotation XML
            annotation_candidates = [
                p for p in annotation_dir.glob("*.xml")
                if f"-{polarization.lower()}-" in p.name.lower()
            ]

            if not annotation_candidates:
                print("Annotation missing:", raster_path.name)
                continue

            annotation_path = annotation_candidates[0]
            xml_root = etree.parse(str(annotation_path))

            # Extract geolocation grid latitude/longitude
            latitudes = []
            longitudes = []

            grid_points = xml_root.xpath(
                "//*[local-name()='geolocationGridPoint']"
            )

            for point in grid_points:
                lat_node = point.xpath("./*[local-name()='latitude']")
                lon_node = point.xpath("./*[local-name()='longitude']")

                if lat_node and lon_node:
                    latitudes.append(float(lat_node[0].text))
                    longitudes.append(float(lon_node[0].text))

            if len(latitudes) < 4:
                print("Insufficient geolocation points:", raster_path.name)
                continue

            # Build geographic footprint from annotation grid
            footprint = MultiPoint(
                list(zip(longitudes, latitudes))
            ).convex_hull

            overlap = footprint.intersection(aoi_geometry)

            # Read raster dimensions; do not assume the TIFF has a CRS
            with rasterio.open(raster_path) as src:
                width = src.width
                height = src.height
                raster_crs = str(src.crs) if src.crs else "Not embedded"

            footprint_records.append({
                "scene": row["scene"],
                "raster": raster_path.name,
                "polarization": polarization,
                "crs": "EPSG:4326 (annotation geolocation grid)",
                "width": width,
                "height": height,
                "aoi_intersects": bool(footprint.intersects(aoi_geometry)),
                "aoi_overlap_area_deg2": overlap.area,
                "raster_path": str(raster_path),
                "annotation_path": str(annotation_path)
            })

            print(
                f"Processed: {raster_path.name} | "
                f"AOI intersection: {footprint.intersects(aoi_geometry)}"
            )

        except Exception as exc:
            print(f"Geolocation failed: {raster_path.name} | {exc}")

footprint_df = pd.DataFrame(footprint_records)

if footprint_df.empty:
    raise RuntimeError(
        "No geolocated measurement rasters found. "
        "Check SAFE annotation XML files."
    )

footprint_df.to_csv(
    DIRS["metadata"] / "sentinel1_raster_footprints.csv",
    index=False
)

print("\n" + "=" * 65)
print("SENTINEL-1 AOI FOOTPRINT ANALYSIS")
print("=" * 65)

display(
    footprint_df[
        ["scene", "raster", "polarization", "width",
         "height", "aoi_intersects"]
    ]
)

print("\nAOI-INTERSECTING SCENES:")
display(
    footprint_df[footprint_df["aoi_intersects"]][
        ["scene", "polarization", "raster"]
    ]
)

print("\nFootprint inventory saved.")
print("STATUS: GEOLOCATION CHECK COMPLETE")



In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 09: COMPATIBLE SCENE PAIR CANDIDATES
# ============================================================

scene_overlap = (
    footprint_df[footprint_df["aoi_intersects"]]
    .groupby("scene")
    .agg(
        intersecting_measurements=("raster", "count"),
        polarizations=("polarization", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

pair_scenes = scene_df.merge(scene_overlap, on="scene", how="inner")

pair_candidates = []

for _, pre in pair_scenes[pair_scenes["event_phase"] == "PRE_EVENT"].iterrows():
    for _, post in pair_scenes[pair_scenes["event_phase"] == "POST_EVENT"].iterrows():

        same_pass = str(pre["pass"]).upper() == str(post["pass"]).upper()
        same_relative_orbit = (
            pd.notna(pre["relative_orbit"])
            and pd.notna(post["relative_orbit"])
            and pre["relative_orbit"] == post["relative_orbit"]
        )
        same_mode = pre["mode"] == post["mode"]
        same_product = pre["product_type"] == post["product_type"]

        delta_days = abs(
            (post["start_time"] - pre["start_time"]).total_seconds()
        ) / 86400

        pair_candidates.append({
            "pre_scene": pre["scene"],
            "post_scene": post["scene"],
            "pre_date": pre["acquisition_date"],
            "post_date": post["acquisition_date"],
            "same_pass": same_pass,
            "same_relative_orbit": same_relative_orbit,
            "same_mode": same_mode,
            "same_product_type": same_product,
            "time_difference_days": round(delta_days, 2),
            "compatible_candidate": (
                same_pass and same_relative_orbit and same_mode and same_product
            )
        })

pair_df = pd.DataFrame(pair_candidates)

if pair_df.empty:
    print("No pre/post pairs found with AOI-intersecting scenes.")
else:
    pair_df = pair_df.sort_values(
        ["compatible_candidate", "time_difference_days"],
        ascending=[False, True]
    )
    display(pair_df)

pair_df.to_csv(
    DIRS["metadata"] / "sentinel1_pair_candidates.csv",
    index=False
)

print("\nPair selection is provisional until geometry and polarization are checked.")


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 10: METADATA CHECKPOINT REPORT
# ============================================================

report = {
    "project": "SARC — Sentinel-1 Event Analysis",
    "study_area": "Rangpo–Majitar, Teesta River Basin, Sikkim",
    "aoi": AOI,
    "analysis_crs": PROJECT_CRS,
    "event_context": "04 October 2023 South Lhonak GLOF and downstream Teesta flood",
    "scene_count": int(len(scene_df)),
    "aoi_intersecting_scene_count": int(
        scene_overlap["scene"].nunique() if not scene_overlap.empty else 0
    ),
    "compatible_pair_count": int(
        pair_df["compatible_candidate"].sum() if not pair_df.empty else 0
    ),
    "metadata_stage": "COMPLETE",
    "radiometric_processing": "NOT YET PERFORMED",
    "ground_truth_validation": "NOT PERFORMED"
}

report_path = DIRS["metadata"] / "SARC_Sentinel1_Metadata_Report.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=4, default=str)

print("=" * 65)
print("SARC SENTINEL-1 METADATA CHECKPOINT")
print("=" * 65)

for key, value in report.items():
    print(f"{key}: {value}")

print("\nReport saved:", report_path)
print("\nNEXT STAGE: CALIBRATION AND TERRAIN CORRECTION")


In [ ]:

# ============================================================
# SARC — SENTINEL-1 EVENT ANALYSIS
# CELL 11: SNAP GPT ENVIRONMENT PREFLIGHT
# ============================================================

import shutil
import subprocess
import os
from pathlib import Path

print("=" * 65)
print("SARC — SNAP PROCESSING PREFLIGHT")
print("=" * 65)

# Storage
disk = shutil.disk_usage("/content")
print(f"\nFree storage: {disk.free / 1024**3:.2f} GB")

# Java
java_path = shutil.which("java")
print("\nJAVA:", java_path or "NOT FOUND")

if java_path:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True
    )
    print((result.stderr or result.stdout).strip())

# SNAP GPT
gpt_path = shutil.which("gpt")
print("\nSNAP GPT:", gpt_path or "NOT FOUND")

if gpt_path:
    result = subprocess.run(
        [gpt_path, "--diag"],
        capture_output=True,
        text=True,
        timeout=60
    )
    print((result.stdout or result.stderr)[:3000])

# Input data
print("\nINPUT SCENES")

for _, row in scene_df.iterrows():
    safe = Path(row["safe_path"])
    print("\n", safe.name)
    print("SAFE exists:", safe.exists())
    print("Manifest exists:", (safe / "manifest.safe").exists())
    print("Annotation XMLs:", len(list((safe / "annotation").glob("*.xml"))))
    print("Measurement TIFFs:", len(list(safe.rglob("measurement/*.tiff"))))

print("\n" + "=" * 65)
print("PREFLIGHT COMPLETE")
print("=" * 65)


In [ ]:

# CELL 12 — SNAP INSTALLATION RECOVERY + LOCATOR

import os
import subprocess
from pathlib import Path

print("=" * 65)
print("SARC — SNAP GPT RECOVERY")
print("=" * 65)

# Check common executable locations
search_roots = [
    Path("/content"),
    Path("/opt"),
    Path("/usr/local"),
    Path("/root"),
    Path.home()
]

matches = []

for root in search_roots:
    if not root.exists():
        continue

    result = subprocess.run(
        ["find", str(root), "-type", "f", "-name", "gpt"],
        capture_output=True,
        text=True,
        timeout=180
    )

    matches.extend(
        Path(p.strip())
        for p in result.stdout.splitlines()
        if p.strip()
    )

# Remove duplicates
matches = list(dict.fromkeys(matches))

if matches:
    GPT = str(matches[0])
    os.chmod(GPT, 0o755)

    test = subprocess.run(
        [GPT, "--diag"],
        capture_output=True,
        text=True,
        timeout=120
    )

    print("\nGPT FOUND:", GPT)
    print("Diagnostic exit code:", test.returncode)
    print((test.stdout or test.stderr)[-2500:])

    if test.returncode != 0:
        raise RuntimeError("GPT found, but diagnostic failed.")

    print("\nSNAP GPT READY")
else:
    print("\nGPT EXECUTABLE NOT FOUND.")

    print("\nPossible installer files:")
    for root in [Path("/content"), Path("/tmp")]:
        if root.exists():
            for p in root.glob("*"):
                if any(x in p.name.lower() for x in
                       ["snap", "esa", "sentinel"]):
                    print(p)

    raise RuntimeError(
        "SNAP installation did not produce a GPT executable. "
        "Share this cell's output before reinstalling."
    )


In [ ]:

# CELL 13 — AOI-FIRST OPTIMIZED SENTINEL-1 PROCESSING

import subprocess
import rasterio
import numpy as np
from pathlib import Path
from shapely.geometry import box, mapping
from rasterio.mask import mask

assert "GPT" in globals() and Path(GPT).exists()
assert "scene_df" in globals() and len(scene_df) == 2

OUT_DIR = DIRS["processed"] / "terrain_corrected"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# AOI with geographic buffer
buffer_deg = 0.03
aoi_buffer = box(
    AOI["west"] - buffer_deg,
    AOI["south"] - buffer_deg,
    AOI["east"] + buffer_deg,
    AOI["north"] + buffer_deg
)
geo_wkt = aoi_buffer.wkt

# Detect optional operator
use_tnr = subprocess.run(
    [GPT, "ThermalNoiseRemoval", "-h"],
    capture_output=True, text=True, timeout=60
).returncode == 0

thermal_node = """
<node id="ThermalNoise">
  <operator>ThermalNoiseRemoval</operator>
  <sources><sourceProduct refid="Orbit"/></sources>
  <parameters/>
</node>
""" if use_tnr else ""

cal_source = "ThermalNoise" if use_tnr else "Orbit"

results = []

for _, row in scene_df.sort_values("start_time").iterrows():

    safe = Path(row["safe_path"])
    name = safe.name.replace(".SAFE", "")

    full_output = OUT_DIR / f"{name}_Sigma0_TC_full.tif"
    aoi_output = OUT_DIR / f"{name}_Sigma0_TC_AOI.tif"
    graph_path = DIRS["temp"] / f"{name}_AOI_graph.xml"

    for p in [full_output, aoi_output]:
        if p.exists():
            p.unlink()

    graph = f"""
<graph id="SARC_AOI_FIRST">
  <version>1.0</version>

  <node id="Read">
    <operator>Read</operator>
    <parameters>
      <file>{safe.as_posix()}</file>
    </parameters>
  </node>

  <node id="Orbit">
    <operator>Apply-Orbit-File</operator>
    <sources><sourceProduct refid="Read"/></sources>
    <parameters>
      <orbitType>Sentinel Precise (Auto Download)</orbitType>
      <polyDegree>3</polyDegree>
      <continueOnFail>true</continueOnFail>
    </parameters>
  </node>

  {thermal_node}

  <node id="Calibration">
    <operator>Calibration</operator>
    <sources><sourceProduct refid="{cal_source}"/></sources>
    <parameters>
      <selectedPolarisations>VV,VH</selectedPolarisations>
      <outputImageInComplex>false</outputImageInComplex>
      <outputImageScaleInDb>false</outputImageScaleInDb>
      <createGammaBand>false</createGammaBand>
      <createBetaBand>false</createBetaBand>
      <outputSigmaBand>true</outputSigmaBand>
    </parameters>
  </node>

  <node id="Subset">
    <operator>Subset</operator>
    <sources><sourceProduct refid="Calibration"/></sources>
    <parameters>
      <copyMetadata>true</copyMetadata>
      <geoRegion>{geo_wkt}</geoRegion>
      <fullSwath>false</fullSwath>
    </parameters>
  </node>

  <node id="TerrainCorrection">
    <operator>Terrain-Correction</operator>
    <sources><sourceProduct refid="Subset"/></sources>
    <parameters>
      <demName>SRTM 1Sec HGT</demName>
      <pixelSpacingInMeter>10.0</pixelSpacingInMeter>
      <mapProjection>EPSG:32645</mapProjection>
      <saveDEM>false</saveDEM>
      <saveLatLon>false</saveLatLon>
      <saveIncidenceAngleFromEllipsoid>false</saveIncidenceAngleFromEllipsoid>
      <saveLocalIncidenceAngle>false</saveLocalIncidenceAngle>
      <saveProjectedLocalIncidenceAngle>false</saveProjectedLocalIncidenceAngle>
      <saveSelectedSourceBand>true</saveSelectedSourceBand>
    </parameters>
  </node>

  <node id="Write">
    <operator>Write</operator>
    <sources><sourceProduct refid="TerrainCorrection"/></sources>
    <parameters>
      <file>{full_output.as_posix()}</file>
      <formatName>GeoTIFF-BigTIFF</formatName>
    </parameters>
  </node>
</graph>
"""

    graph_path.parent.mkdir(parents=True, exist_ok=True)
    graph_path.write_text(graph)

    print("\n" + "=" * 60)
    print("PROCESSING:", name, flush=True)
    print("AOI subset buffer:", buffer_deg, "degrees", flush=True)

    # Stream SNAP logs instead of waiting silently
    process = subprocess.Popen(
        [GPT, str(graph_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    log_tail = []
    for line in process.stdout:
        line = line.rstrip()
        if line:
            print(line, flush=True)
            log_tail.append(line)
            log_tail = log_tail[-60:]

    return_code = process.wait()

    if return_code != 0 or not full_output.exists():
        raise RuntimeError(
            f"SNAP failed for {name}:\n" + "\n".join(log_tail)
        )

    # Exact common NISAR AOI clip
    with rasterio.open(full_output) as src:
        geometry = aoi_gdf.to_crs(src.crs).geometry.iloc[0]
        clipped, transform = mask(
            src, [mapping(geometry)], crop=True, filled=True
        )

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=clipped.shape[1],
            width=clipped.shape[2],
            transform=transform,
            compress="DEFLATE",
            tiled=True,
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(aoi_output, "w", **profile) as dst:
            dst.write(clipped)
            for i, desc in enumerate(src.descriptions, 1):
                if desc:
                    dst.set_band_description(i, desc)

    # Integrated QA
    with rasterio.open(aoi_output) as check:
        data = check.read(masked=True)
        valid = int(np.ma.count(data))

        if valid == 0 or check.crs.to_string() != "EPSG:32645":
            raise RuntimeError("Output QA failed.")

        results.append(aoi_output)

        print("\nSCENE PASSED:", name)
        print("Dimensions:", check.width, "x", check.height)
        print("Bands:", check.descriptions)
        print("Valid pixels:", valid)
        print("Output:", aoi_output)

print("\n" + "=" * 65)
print("SARC — AOI-FIRST PROCESSING COMPLETE")
print("Successful scenes:", len(results), "/ 2")
print("Output folder:", OUT_DIR)


In [ ]:

# CELL 14 — SENTINEL-1 BACKSCATTER CHANGE ANALYSIS

import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from rasterio.warp import reproject, Resampling

OUT = DIRS["outputs"] / "backscatter_change"
OUT.mkdir(parents=True, exist_ok=True)

files = sorted(
    (DIRS["processed"] / "terrain_corrected").glob("*_Sigma0_TC_AOI.tif")
)

assert len(files) == 2, f"Expected 2 scenes, found {len(files)}"

# Identify dates from filenames
pre_file = next(p for p in files if "20231002" in p.name)
post_file = next(p for p in files if "20231014" in p.name)

# Reference grid = 02 October
with rasterio.open(pre_file) as pre:
    ref_profile = pre.profile.copy()
    ref_transform = pre.transform
    ref_crs = pre.crs
    ref_shape = (pre.height, pre.width)
    pre_data = pre.read().astype("float32")
    pre_nodata = pre.nodata
    pre_tags = pre.tags()
    pre_band_tags = [pre.tags(i) for i in range(1, pre.count + 1)]

with rasterio.open(post_file) as post:
    post_data = post.read().astype("float32")
    post_nodata = post.nodata
    post_transform = post.transform
    post_crs = post.crs
    post_shape = (post.height, post.width)
    post_tags = post.tags()
    post_band_tags = [post.tags(i) for i in range(1, post.count + 1)]

print("PRE metadata:", pre_tags)
print("POST metadata:", post_tags)
print("PRE band tags:", pre_band_tags)
print("POST band tags:", post_band_tags)

assert pre_data.shape[0] >= 2 and post_data.shape[0] >= 2, \
    "Expected calibrated VV and VH bands."

# Selected polarisation order in SNAP graph: VV, VH
# Reproject post-event bands to the pre-event grid
post_aligned = np.full(
    (2, *ref_shape), np.nan, dtype="float32"
)

for b in range(2):
    source = post_data[b].copy()

    if post_nodata is not None:
        source[source == post_nodata] = np.nan

    reproject(
        source=source,
        destination=post_aligned[b],
        src_transform=post_transform,
        src_crs=post_crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        src_nodata=np.nan,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear
    )

# Convert linear Sigma0 to dB
pre_db = np.full((2, *ref_shape), np.nan, dtype="float32")
post_db = np.full((2, *ref_shape), np.nan, dtype="float32")

for b in range(2):
    before = pre_data[b]
    after = post_aligned[b]

    valid_before = np.isfinite(before) & (before > 0)
    if pre_nodata is not None:
        valid_before &= before != pre_nodata

    valid_after = np.isfinite(after) & (after > 0)

    pre_db[b][valid_before] = 10 * np.log10(before[valid_before])
    post_db[b][valid_after] = 10 * np.log10(after[valid_after])

# Difference = post-event minus pre-event
delta_db = post_db - pre_db

valid_common = np.isfinite(pre_db) & np.isfinite(post_db)

# Export change rasters
profile = ref_profile.copy()
profile.update(
    driver="GTiff",
    count=2,
    dtype="float32",
    nodata=-9999.0,
    compress="DEFLATE",
    tiled=True
)

delta_path = OUT / "SARC_S1_Backscatter_Change_dB.tif"

with rasterio.open(delta_path, "w", **profile) as dst:
    for b, pol in enumerate(["VV", "VH"]):
        output = np.where(
            np.isfinite(delta_db[b]), delta_db[b], -9999
        ).astype("float32")
        dst.write(output, b + 1)
        dst.set_band_description(b + 1, f"{pol}_PostMinusPre_dB")

# Statistics
records = []

for b, pol in enumerate(["VV", "VH"]):
    valid = valid_common[b]
    values = delta_db[b][valid]

    if values.size == 0:
        raise RuntimeError(f"No common valid pixels for {pol}")

    records.append({
        "polarization": pol,
        "valid_pixels": int(values.size),
        "pre_mean_db": float(np.nanmean(pre_db[b])),
        "post_mean_db": float(np.nanmean(post_db[b])),
        "change_mean_db": float(np.mean(values)),
        "change_median_db": float(np.median(values)),
        "change_std_db": float(np.std(values)),
        "change_p05_db": float(np.percentile(values, 5)),
        "change_p95_db": float(np.percentile(values, 95)),
        "increase_pixels_pct": float(np.mean(values > 0) * 100),
        "decrease_pixels_pct": float(np.mean(values < 0) * 100)
    })

stats_df = pd.DataFrame(records)
stats_df.to_csv(OUT / "SARC_S1_Backscatter_Change_Statistics.csv", index=False)

# Research map
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for b, pol in enumerate(["VV", "VH"]):
    panels = [
        (pre_db[b], f"{pol} — 02 Oct 2023", "gray", -30, 5),
        (post_db[b], f"{pol} — 14 Oct 2023", "gray", -30, 5),
        (delta_db[b], f"{pol} — Post minus Pre (dB)", "RdBu_r", -6, 6)
    ]

    for j, (array, title, cmap, vmin, vmax) in enumerate(panels):
        axes[b, j].imshow(
            np.ma.masked_invalid(array),
            cmap=cmap, vmin=vmin, vmax=vmax
        )
        axes[b, j].set_title(title)
        axes[b, j].axis("off")

fig.suptitle(
    "SARC — Sentinel-1 Multi-Temporal Backscatter Change\n"
    "Rangpo–Majitar | Teesta River Basin | October 2023",
    fontsize=14
)
plt.tight_layout()

map_path = OUT / "SARC_S1_Backscatter_Change_Map.png"
plt.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()

# Integrated QA
assert np.isfinite(delta_db).any(), "No valid change pixels."
assert delta_path.exists() and map_path.exists()

print("\n" + "=" * 65)
print("SARC — BACKSCATTER CHANGE ANALYSIS COMPLETE")
print("=" * 65)
print(stats_df.to_string(index=False))
print("\nChange GeoTIFF:", delta_path)
print("Statistics CSV:", OUT / "SARC_S1_Backscatter_Change_Statistics.csv")
print("Research map:", map_path)
print("Common valid pixels — VV:", int(valid_common[0].sum()))
print("Common valid pixels — VH:", int(valid_common[1].sum()))


In [ ]:

# CELL 15 — SPECKLE-AWARE CHANGE DETECTION + ANOMALY INVENTORY

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from scipy.ndimage import median_filter, label
from pathlib import Path

assert "pre_db" in globals() and "post_db" in globals()
assert pre_db.shape == post_db.shape
assert pre_db.shape[0] >= 2

OUT = DIRS["outputs"] / "backscatter_change"
OUT.mkdir(parents=True, exist_ok=True)

polarizations = ["VV", "VH"]
filtered_delta = np.full_like(pre_db, np.nan, dtype="float32")
anomaly_masks = {}
component_records = []
summary_records = []

# Pixel area from projected reference grid
pixel_area = abs(
    ref_transform.a * ref_transform.e -
    ref_transform.b * ref_transform.d
)

for b, pol in enumerate(polarizations):

    valid = np.isfinite(pre_db[b]) & np.isfinite(post_db[b])

    if valid.sum() == 0:
        raise RuntimeError(f"No common valid pixels for {pol}")

    # Median filtering with invalid pixels temporarily filled
    pre_filled = np.where(
        np.isfinite(pre_db[b]), pre_db[b],
        np.nanmedian(pre_db[b])
    )
    post_filled = np.where(
        np.isfinite(post_db[b]), post_db[b],
        np.nanmedian(post_db[b])
    )

    pre_smooth = median_filter(pre_filled, size=3)
    post_smooth = median_filter(post_filled, size=3)

    change = post_smooth - pre_smooth
    change[~valid] = np.nan
    filtered_delta[b] = change

    values = change[valid]

    # Robust distribution-based thresholds
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    robust_sigma = 1.4826 * mad

    if robust_sigma <= 0:
        raise RuntimeError(f"Degenerate MAD for {pol}")

    lower = median - 2.5 * robust_sigma
    upper = median + 2.5 * robust_sigma

    increase = valid & (change > upper)
    decrease = valid & (change < lower)

    anomaly_masks[pol] = {
        "increase": increase,
        "decrease": decrease,
        "median": median,
        "mad": mad,
        "robust_sigma": robust_sigma,
        "lower_threshold": lower,
        "upper_threshold": upper
    }

    # Connected component inventory; retain >= 5 pixels
    for direction, binary in [
        ("Backscatter_Increase", increase),
        ("Backscatter_Decrease", decrease)
    ]:
        labels, count = label(binary)
        sizes = np.bincount(labels.ravel())

        retained = 0
        total_pixels = 0

        for component_id in range(1, count + 1):
            size = int(sizes[component_id])

            if size < 5:
                continue

            rows, cols = np.where(labels == component_id)
            center_row = float(rows.mean())
            center_col = float(cols.mean())

            x, y = ref_transform * (
                center_col + 0.5, center_row + 0.5
            )

            component_records.append({
                "polarization": pol,
                "change_type": direction,
                "component_id": component_id,
                "pixel_count": size,
                "area_m2": size * pixel_area,
                "centroid_x": x,
                "centroid_y": y,
                "mean_change_db": float(
                    np.mean(change[labels == component_id])
                )
            })

            retained += 1
            total_pixels += size

        summary_records.append({
            "polarization": pol,
            "change_type": direction,
            "threshold_db": upper if direction == "Backscatter_Increase" else lower,
            "candidate_pixels": int(binary.sum()),
            "components_total": int(count),
            "components_ge_5_pixels": retained,
            "retained_candidate_area_m2": total_pixels * pixel_area
        })

    summary_records.append({
        "polarization": pol,
        "change_type": "Robust_Distribution",
        "median_change_db": median,
        "MAD_db": mad,
        "robust_sigma_db": robust_sigma,
        "lower_threshold_db": lower,
        "upper_threshold_db": upper
    })

# Export filtered change raster
profile = ref_profile.copy()
profile.update(
    driver="GTiff",
    count=2,
    dtype="float32",
    nodata=-9999.0,
    compress="DEFLATE",
    tiled=True
)

filtered_path = OUT / "SARC_S1_SpeckleFiltered_Change_dB.tif"

with rasterio.open(filtered_path, "w", **profile) as dst:
    for b, pol in enumerate(polarizations):
        output = np.where(
            np.isfinite(filtered_delta[b]),
            filtered_delta[b],
            -9999
        ).astype("float32")

        dst.write(output, b + 1)
        dst.set_band_description(
            b + 1, f"{pol}_Filtered_PostMinusPre_dB"
        )

# Export anomaly inventory and statistics
components_df = pd.DataFrame(component_records)
summary_df = pd.DataFrame(summary_records)

components_path = OUT / "SARC_S1_Change_Component_Inventory.csv"
summary_path = OUT / "SARC_S1_Robust_Change_Statistics.csv"

components_df.to_csv(components_path, index=False)
summary_df.to_csv(summary_path, index=False)

# Research visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for b, pol in enumerate(polarizations):

    change = filtered_delta[b]
    info = anomaly_masks[pol]

    axes[b, 0].imshow(
        np.ma.masked_invalid(change),
        cmap="RdBu_r", vmin=-6, vmax=6
    )
    axes[b, 0].set_title(f"{pol} — Median-filtered change (dB)")
    axes[b, 0].axis("off")

    candidate_map = np.zeros(change.shape, dtype="uint8")
    candidate_map[info["increase"]] = 1
    candidate_map[info["decrease"]] = 2
    candidate_map[~np.isfinite(change)] = 255

    axes[b, 1].imshow(
        np.ma.masked_where(candidate_map == 255, candidate_map),
        cmap="viridis", vmin=0, vmax=2
    )
    axes[b, 1].set_title(
        f"{pol} — Candidate increase / decrease"
    )
    axes[b, 1].axis("off")

fig.suptitle(
    "SARC — Sentinel-1 Speckle-Aware Change Inventory\n"
    "Rangpo–Majitar | 02–14 October 2023",
    fontsize=14
)
plt.tight_layout()

map_path = OUT / "SARC_S1_SpeckleAware_Change_Map.png"
plt.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()

# Integrated QA
assert filtered_path.exists()
assert components_path.exists()
assert summary_path.exists()
assert np.isfinite(filtered_delta).any()

print("\n" + "=" * 65)
print("SARC — SPECKLE-AWARE CHANGE ANALYSIS COMPLETE")
print("=" * 65)

print("\nROBUST STATISTICS AND CANDIDATE INVENTORY")
print(summary_df.to_string(index=False))

print("\nCOMPONENTS RETAINED:", len(components_df))
print("Pixel area:", round(pixel_area, 2), "m²")
print("\nFiltered GeoTIFF:", filtered_path)
print("Component inventory:", components_path)
print("Statistics:", summary_path)
print("Research map:", map_path)


In [ ]:

# CELL 16 — VV/VH CROSS-POLARIZATION AGREEMENT

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from scipy.ndimage import label
from pathlib import Path

OUT = DIRS["outputs"] / "backscatter_change"
OUT.mkdir(parents=True, exist_ok=True)

assert "anomaly_masks" in globals()
assert "filtered_delta" in globals()

vv = anomaly_masks["VV"]
vh = anomaly_masks["VH"]

# Same-direction anomalies detected by both polarizations
common_increase = vv["increase"] & vh["increase"]
common_decrease = vv["decrease"] & vh["decrease"]

# Opposite-direction detections
polarization_disagreement = (
    (vv["increase"] & vh["decrease"]) |
    (vv["decrease"] & vh["increase"])
)

# Consensus classification:
# 0 = No consensus anomaly
# 1 = Common backscatter increase
# 2 = Common backscatter decrease
consensus = np.zeros(common_increase.shape, dtype="uint8")
consensus[common_increase] = 1
consensus[common_decrease] = 2

valid = np.isfinite(filtered_delta[0]) & np.isfinite(filtered_delta[1])
consensus[~valid] = 255

pixel_area = abs(
    ref_transform.a * ref_transform.e -
    ref_transform.b * ref_transform.d
)

# Connected components using 8-neighbour connectivity
structure = np.ones((3, 3), dtype=np.uint8)

component_records = []

for class_id, class_name in [
    (1, "Common_Increase"),
    (2, "Common_Decrease")
]:
    binary = consensus == class_id
    labels, count = label(binary, structure=structure)
    sizes = np.bincount(labels.ravel())

    for component_id in range(1, count + 1):
        size = int(sizes[component_id])

        if size < 5:
            continue

        rows, cols = np.where(labels == component_id)

        center_row = float(rows.mean())
        center_col = float(cols.mean())

        x, y = ref_transform * (
            center_col + 0.5, center_row + 0.5
        )

        component_records.append({
            "change_class": class_name,
            "component_id": component_id,
            "pixel_count": size,
            "area_m2": size * pixel_area,
            "centroid_x": x,
            "centroid_y": y,
            "mean_VV_change_db": float(
                np.nanmean(filtered_delta[0][labels == component_id])
            ),
            "mean_VH_change_db": float(
                np.nanmean(filtered_delta[1][labels == component_id])
            )
        })

components_df = pd.DataFrame(component_records)

# Agreement statistics
vv_candidates = vv["increase"] | vv["decrease"]
vh_candidates = vh["increase"] | vh["decrease"]
candidate_union = vv_candidates | vh_candidates
candidate_intersection = common_increase | common_decrease

union_count = int(candidate_union.sum())
agreement_count = int(candidate_intersection.sum())

agreement_pct = (
    100 * agreement_count / union_count
    if union_count else 0
)

summary = pd.DataFrame([{
    "VV_candidate_pixels": int(vv_candidates.sum()),
    "VH_candidate_pixels": int(vh_candidates.sum()),
    "common_increase_pixels": int(common_increase.sum()),
    "common_decrease_pixels": int(common_decrease.sum()),
    "opposite_direction_pixels": int(polarization_disagreement.sum()),
    "candidate_union_pixels": union_count,
    "same_direction_agreement_pixels": agreement_count,
    "same_direction_agreement_pct_of_union": agreement_pct,
    "common_candidate_area_m2": agreement_count * pixel_area,
    "retained_components_ge_5_pixels": len(components_df)
}])

# Save consensus raster
consensus_path = OUT / "SARC_S1_VV_VH_Consensus_Change.tif"

profile = ref_profile.copy()
profile.update(
    driver="GTiff",
    count=1,
    dtype="uint8",
    nodata=255,
    compress="DEFLATE",
    tiled=True
)

with rasterio.open(consensus_path, "w", **profile) as dst:
    dst.write(consensus, 1)
    dst.set_band_description(
        1, "VV_VH_SameDirection_Change_Consensus"
    )

components_path = OUT / "SARC_S1_VV_VH_Consensus_Components.csv"
summary_path = OUT / "SARC_S1_VV_VH_Agreement_Statistics.csv"

components_df.to_csv(components_path, index=False)
summary.to_csv(summary_path, index=False)

# Research visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

axes[0].imshow(
    np.ma.masked_where(consensus == 255, consensus),
    cmap="viridis", vmin=0, vmax=2
)
axes[0].set_title("VV–VH Same-direction Consensus")

axes[1].imshow(
    np.ma.masked_invalid(filtered_delta[0]),
    cmap="RdBu_r", vmin=-6, vmax=6
)
axes[1].set_title("VV Filtered Change (dB)")

axes[2].imshow(
    np.ma.masked_invalid(filtered_delta[1]),
    cmap="RdBu_r", vmin=-6, vmax=6
)
axes[2].set_title("VH Filtered Change (dB)")

for ax in axes:
    ax.axis("off")

fig.suptitle(
    "SARC — Sentinel-1 Cross-Polarization Agreement\n"
    "Rangpo–Majitar | October 2023",
    fontsize=14
)
plt.tight_layout()

map_path = OUT / "SARC_S1_VV_VH_Consensus_Map.png"
plt.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()

# Integrated QA
assert consensus_path.exists()
assert summary_path.exists()
assert np.isfinite(filtered_delta).any()

print("\n" + "=" * 65)
print("SARC — CROSS-POLARIZATION ANALYSIS COMPLETE")
print("=" * 65)
print(summary.to_string(index=False))

print("\nCONSENSUS COMPONENTS")
print(components_df.head(20).to_string(index=False))

print("\nConsensus GeoTIFF:", consensus_path)
print("Component inventory:", components_path)
print("Statistics:", summary_path)
print("Research map:", map_path)


In [ ]:

# CELL 17 — VV/VH CONSENSUS GEOSPATIAL INVENTORY

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from pathlib import Path
from scipy.ndimage import label
from rasterio.features import shapes
from shapely.geometry import shape

OUT = DIRS["outputs"] / "backscatter_change"
OUT.mkdir(parents=True, exist_ok=True)

assert "consensus" in globals()
assert "filtered_delta" in globals()
assert "ref_transform" in globals()
assert "ref_crs" in globals()

records = []
candidate_id = 0

# Process increase and decrease separately
for class_value, class_name in [
    (1, "Common_Increase"),
    (2, "Common_Decrease")
]:
    binary = consensus == class_value

    labels, count = label(
        binary,
        structure=np.ones((3, 3), dtype=np.uint8)
    )

    sizes = np.bincount(labels.ravel())

    for geom_mapping, component_id in shapes(
        labels.astype("int32"),
        mask=labels > 0,
        transform=ref_transform,
        connectivity=8
    ):
        component_id = int(component_id)
        pixel_count = int(sizes[component_id])

        if pixel_count < 5:
            continue

        geometry = shape(geom_mapping)
        component_mask = labels == component_id

        candidate_id += 1

        centroid = geometry.centroid
        vv_values = filtered_delta[0][component_mask]
        vh_values = filtered_delta[1][component_mask]

        records.append({
            "candidate_id": candidate_id,
            "change_class": class_name,
            "component_id": component_id,
            "pixel_count": pixel_count,
            "area_m2": float(geometry.area),
            "area_ha": float(geometry.area / 10000),
            "centroid_x": float(centroid.x),
            "centroid_y": float(centroid.y),
            "VV_mean_change_db": float(np.nanmean(vv_values)),
            "VH_mean_change_db": float(np.nanmean(vh_values)),
            "VV_median_change_db": float(np.nanmedian(vv_values)),
            "VH_median_change_db": float(np.nanmedian(vh_values)),
            "geometry": geometry
        })

if not records:
    raise RuntimeError("No consensus components with >=5 pixels.")

candidate_gdf = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs=ref_crs
)

# Rank by spatial area, not flood severity
candidate_gdf = candidate_gdf.sort_values(
    "area_m2", ascending=False
).reset_index(drop=True)

candidate_gdf["area_rank"] = np.arange(
    1, len(candidate_gdf) + 1
)

# Export geospatial formats
gpkg_path = OUT / "SARC_S1_Consensus_Candidate_Inventory.gpkg"
geojson_path = OUT / "SARC_S1_Consensus_Candidate_Inventory.geojson"
csv_path = OUT / "SARC_S1_Consensus_Candidate_Inventory.csv"

candidate_gdf.to_file(
    gpkg_path,
    layer="consensus_candidates",
    driver="GPKG"
)

candidate_gdf.to_crs("EPSG:4326").to_file(
    geojson_path,
    driver="GeoJSON"
)

candidate_gdf.drop(columns="geometry").to_csv(
    csv_path, index=False
)

# Spatial overview map
fig, ax = plt.subplots(figsize=(10, 9))

candidate_gdf.plot(
    ax=ax,
    column="change_class",
    categorical=True,
    legend=True,
    edgecolor="black",
    linewidth=0.15,
    alpha=0.85
)

ax.set_title(
    "SARC — VV/VH Consensus Change Candidates\n"
    "Rangpo–Majitar | October 2023"
)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")

plt.tight_layout()

map_path = OUT / "SARC_S1_Consensus_Candidate_Spatial_Map.png"
plt.savefig(map_path, dpi=300, bbox_inches="tight")
plt.show()

# Integrated QA and inventory summary
assert candidate_gdf.crs is not None
assert gpkg_path.exists() and geojson_path.exists()
assert candidate_gdf.geometry.is_valid.all()

print("\n" + "=" * 65)
print("SARC — GEOSPATIAL CANDIDATE INVENTORY COMPLETE")
print("=" * 65)

print("Total retained candidates:", len(candidate_gdf))
print("Common increase:", int(
    (candidate_gdf.change_class == "Common_Increase").sum()
))
print("Common decrease:", int(
    (candidate_gdf.change_class == "Common_Decrease").sum()
))
print("Total candidate area (ha):",
      round(candidate_gdf.area.sum() / 10000, 4))

print("\nTOP 15 SPATIAL CANDIDATES")
print(
    candidate_gdf.drop(columns="geometry")
    .head(15).to_string(index=False)
)

print("\nGeoPackage:", gpkg_path)
print("GeoJSON:", geojson_path)
print("CSV:", csv_path)
print("Map:", map_path)


In [ ]:

# CELL 17A — REPAIR INVALID GEOMETRIES AND RE-EXPORT

import geopandas as gpd
import pandas as pd
from pathlib import Path

OUT = DIRS["outputs"] / "backscatter_change"

# Repair invalid geometries
invalid_before = int((~candidate_gdf.geometry.is_valid).sum())

try:
    from shapely import make_valid
    candidate_gdf["geometry"] = candidate_gdf.geometry.apply(make_valid)
except ImportError:
    candidate_gdf["geometry"] = candidate_gdf.geometry.buffer(0)

# Remove empty or unusable geometries
candidate_gdf = candidate_gdf[
    candidate_gdf.geometry.notna() &
    ~candidate_gdf.geometry.is_empty
].copy()

# Keep polygonal geometry only
candidate_gdf["geometry"] = candidate_gdf.geometry.apply(
    lambda g: g if g.geom_type in ("Polygon", "MultiPolygon")
    else g.buffer(0)
)

candidate_gdf = candidate_gdf[
    candidate_gdf.geometry.is_valid &
    ~candidate_gdf.geometry.is_empty
].copy()

# Recalculate areas after repair
candidate_gdf["area_m2"] = candidate_gdf.geometry.area
candidate_gdf["area_ha"] = candidate_gdf["area_m2"] / 10000

# Export repaired files
gpkg_path = OUT / "SARC_S1_Consensus_Candidate_Inventory.gpkg"
geojson_path = OUT / "SARC_S1_Consensus_Candidate_Inventory.geojson"
csv_path = OUT / "SARC_S1_Consensus_Candidate_Inventory.csv"

candidate_gdf.to_file(
    gpkg_path,
    layer="consensus_candidates",
    driver="GPKG",
    mode="w"
)

candidate_gdf.to_crs("EPSG:4326").to_file(
    geojson_path,
    driver="GeoJSON"
)

candidate_gdf.drop(columns="geometry").to_csv(
    csv_path, index=False
)

# Integrated validation
assert candidate_gdf.crs is not None
assert candidate_gdf.geometry.is_valid.all()
assert gpkg_path.exists() and geojson_path.exists()

print("=" * 60)
print("SARC — GEOMETRY REPAIR COMPLETE")
print("=" * 60)
print("Invalid geometries before repair:", invalid_before)
print("Valid candidate geometries:", len(candidate_gdf))
print("Remaining invalid geometries:",
      int((~candidate_gdf.geometry.is_valid).sum()))
print("GeoPackage:", gpkg_path)
print("GeoJSON:", geojson_path)
print("CSV:", csv_path)


In [ ]:

# CELL 19 — FINAL SARC SENTINEL-1 RESEARCH PACKAGE

import json
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

BASE = Path("/content/SARC_Sentinel1")
OUT = DIRS["outputs"] / "backscatter_change"
PACKAGE_DIR = BASE / "final_research_package"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# Required core outputs only
required = [
    OUT / "SARC_S1_SpeckleFiltered_Change_dB.tif",
    OUT / "SARC_S1_VV_VH_Consensus_Change.tif",
    OUT / "SARC_S1_Consensus_Candidate_Inventory.gpkg",
    OUT / "SARC_S1_Consensus_Candidate_Inventory.geojson",
    OUT / "SARC_S1_Consensus_Candidate_Inventory.csv"
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing core outputs:\n" + "\n".join(missing)
    )

assert candidate_gdf.geometry.is_valid.all()
assert len(candidate_gdf) > 0

# Scene metadata
scene_inventory = []

for _, row in scene_df.iterrows():
    scene_inventory.append({
        "scene": str(row["scene"]),
        "platform": str(row["platform"]),
        "mode": str(row["mode"]),
        "product_type": str(row["product_type"]),
        "acquisition_time": str(row["start_time"]),
        "orbit_pass": str(row["pass"]),
        "relative_orbit": str(row["relative_orbit"]),
        "polarizations": str(row["polarizations"])
    })

# Candidate statistics
candidate_statistics = {
    "total_consensus_candidates": int(len(candidate_gdf)),
    "common_increase_candidates": int(
        (candidate_gdf["change_class"] == "Common_Increase").sum()
    ),
    "common_decrease_candidates": int(
        (candidate_gdf["change_class"] == "Common_Decrease").sum()
    ),
    "total_candidate_area_ha": float(candidate_gdf["area_ha"].sum()),
    "largest_candidate_area_ha": float(candidate_gdf["area_ha"].max()),
    "geometry_valid": bool(candidate_gdf.geometry.is_valid.all())
}

# Optional river analysis: report actual status, do not fabricate results
river_csv = OUT / "SARC_S1_River_Proximity_Inventory.csv"
river_gpkg = OUT / "SARC_S1_River_Proximity_Analysis.gpkg"

river_status = (
    "COMPLETED"
    if river_csv.exists() and river_gpkg.exists()
    else "NOT PERFORMED — OSM Overpass returned HTTP 406/429"
)

report = {
    "project": "SARC — Sentinel-1 Multi-Temporal Event Analysis",
    "study_area": "Rangpo–Majitar, Teesta River Basin, Sikkim, India",
    "event_context": (
        "04 October 2023 South Lhonak GLOF and downstream "
        "Teesta River flood context"
    ),
    "analysis_period": "02–14 October 2023",
    "acquisition_interval_days": 12,
    "aoi": AOI,
    "analysis_crs": PROJECT_CRS,
    "input_scene_count": int(len(scene_df)),
    "input_scenes": scene_inventory,

    "processing": {
        "processor": "ESA SNAP GPT",
        "product": "Sentinel-1 IW GRDH",
        "polarizations": ["VV", "VH"],
        "radiometric_quantity": "Sigma0 backscatter",
        "terrain_correction": "Range-Doppler Terrain Correction",
        "change_method": "Post-event minus pre-event backscatter in dB",
        "speckle_suppression": "3x3 median filtering",
        "anomaly_method": "Median and MAD-based exploratory thresholds",
        "cross_polarization": "Same-direction VV/VH agreement",
        "minimum_component_size_pixels": 5
    },

    "candidate_statistics": candidate_statistics,

    "optional_river_proximity_analysis": {
        "status": river_status,
        "reason": (
            "Public OSM Overpass endpoint returned HTTP 406 and 429. "
            "No river-distance statistics are reported."
        )
    },

    "validation_status": {
        "ground_truth_validation": "NOT PERFORMED",
        "independent_flood_extent_validation": "NOT PERFORMED",
        "event_causality": "NOT ESTABLISHED",
        "operational_flood_classification": "NOT PERFORMED"
    },

    "interpretation_limitations": [
        "Detected regions are exploratory radar-change candidates, not confirmed flood extent.",
        "VV/VH agreement is not independent validation.",
        "Terrain, vegetation, soil moisture, speckle and residual registration effects may influence observed changes.",
        "Two acquisitions cannot establish the complete event evolution.",
        "River proximity analysis was not completed."
    ],

    "generated_utc": datetime.now(timezone.utc).isoformat()
}

report_path = PACKAGE_DIR / "SARC_Sentinel1_Final_Research_Report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# README
readme = f"""
# SARC — Sentinel-1 Multi-Temporal Event Analysis

## Study Area
Rangpo–Majitar, Teesta River Basin, Sikkim, India.

## Event Context
04 October 2023 South Lhonak GLOF and downstream Teesta flood.

## Acquisition Pair
- Pre-event: 02 October 2023
- Post-event: 14 October 2023
- Interval: 12 days
- Platform: Sentinel-1A
- Product: IW GRDH
- Polarizations: VV and VH

## AOI
- North: {AOI['north']} N
- South: {AOI['south']} N
- West: {AOI['west']} E
- East: {AOI['east']} E
- CRS: {PROJECT_CRS}

## Processing
1. SAFE metadata extraction and acquisition-pair selection.
2. Annotation-based geolocation footprint assessment.
3. SNAP radiometric calibration and terrain correction.
4. AOI extraction and backscatter comparison.
5. VV/VH post-minus-pre change analysis.
6. 3x3 median filtering.
7. MAD-based exploratory anomaly identification.
8. VV/VH same-direction consensus.
9. Connected-component candidate inventory.
10. Geometry repair and GIS export.

## Candidate Inventory
- Total candidates: {len(candidate_gdf)}
- Common increase: {candidate_statistics['common_increase_candidates']}
- Common decrease: {candidate_statistics['common_decrease_candidates']}
- Candidate polygon area: {candidate_statistics['total_candidate_area_ha']:.4f} ha

## Optional River Analysis
Status: {river_status}

OSM Overpass requests returned HTTP 406/429. River proximity
statistics have not been included.

## Scientific Limitations
These are exploratory radar backscatter change candidates,
not confirmed flood pixels or validated flood extent.

VV/VH agreement is not independent validation. Ground-truth
validation, independent flood-extent comparison and event-causality
assessment have not been performed.
"""

readme_path = PACKAGE_DIR / "README_SARC_Sentinel1.md"
readme_path.write_text(readme.strip() + "\n", encoding="utf-8")

# Collect core outputs, maps, metadata and AOI products
files_to_package = []

for folder in [OUT, DIRS["metadata"]]:
    if folder.exists():
        files_to_package.extend(
            p for p in folder.rglob("*") if p.is_file()
        )

for p in DIRS["processed"].rglob("*"):
    if p.is_file() and p.name.endswith("_AOI.tif"):
        files_to_package.append(p)

files_to_package.extend([report_path, readme_path])
files_to_package = list(dict.fromkeys(files_to_package))

package_path = BASE / "SARC_Sentinel1_Final_Research_Package.zip"

with zipfile.ZipFile(
    package_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in files_to_package:
        try:
            zf.write(path, arcname=str(path.relative_to(BASE)))
        except ValueError:
            continue

# Integrated QA
assert report_path.exists()
assert readme_path.exists()
assert package_path.exists() and package_path.stat().st_size > 0
assert candidate_gdf.geometry.is_valid.all()

print("\n" + "=" * 65)
print("SARC — SENTINEL-1 FINAL RESEARCH PACKAGE COMPLETE")
print("=" * 65)
print("Scenes:", len(scene_df))
print("Consensus candidates:", len(candidate_gdf))
print("Candidate area (ha):", round(candidate_gdf.area.sum()/10000, 4))
print("River analysis:", river_status)
print("Files packaged:", len(files_to_package))
print("Package size (MB):", round(package_path.stat().st_size/1024**2, 2))
print("\nFinal report:", report_path)
print("README:", readme_path)
print("ZIP package:", package_path)


In [ ]:

from pathlib import Path
from IPython.display import display, Image, Markdown

BASE = Path("/content/SARC_Sentinel1")
folders = [BASE / "outputs", BASE / "final_research_package"]

images = sorted({
    p for folder in folders if folder.exists()
    for p in folder.rglob("*")
    if p.is_file() and p.suffix.lower() in [".png", ".jpg", ".jpeg"]
})

print(f"Total visual outputs found: {len(images)}")

for i, p in enumerate(images, 1):
    display(Markdown(f"### {i}. {p.name}"))
    display(Image(filename=str(p)))


In [ ]:

from pathlib import Path
import shutil
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/SARC_Sentinel1")
DEST = Path("/content/drive/MyDrive/SARC/Sentinel1")
DEST.mkdir(parents=True, exist_ok=True)

# Copy all generated research outputs
for folder in ["outputs", "processed", "metadata"]:
    src = BASE / folder
    if src.exists():
        shutil.copytree(src, DEST / folder, dirs_exist_ok=True)

# Copy final research ZIP
zip_file = BASE / "SARC_Sentinel1_Final_Research_Package.zip"
if zip_file.exists():
    shutil.copy2(zip_file, DEST / zip_file.name)

# Report backup size and file count
saved_files = [p for p in DEST.rglob("*") if p.is_file()]
total_mb = sum(p.stat().st_size for p in saved_files) / (1024 ** 2)

print("=" * 55)
print("SARC SENTINEL-1 — DRIVE BACKUP")
print("=" * 55)
print("Destination:", DEST)
print("Files saved:", len(saved_files))
print(f"Total backup size: {total_mb:.2f} MB")
print("ZIP saved:", (DEST / zip_file.name).exists())


In [ ]:
import os

root = "/content/drive/MyDrive"

for folder, _, files_list in os.walk(root):
    for name in files_list:
        if name.endswith(".ipynb"):
            path = os.path.join(folder, name)
            size = os.path.getsize(path) / (1024 * 1024)
            print(f"{size:8.2f} MB  |  {path}")